[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/13-evaluation.ipynb)

# Module 13: Evaluation

Companion notebook to [Module 13](https://transformers-and-math.drmhse.com/modules/13-evaluation) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
from math import comb
import numpy as np
rng = np.random.default_rng(0)

## Part 1 — Perplexity depends on the tokenizer; bits per byte does not (section 13.1)
A smoothed bigram model scored on the same sentence with two tokenizations: characters, and words.

In [ ]:
train = ('the model reads the text one token at a time and predicts the next token from the tokens before it . '
         'the tokenizer decides how the text is cut into tokens . ') * 20
test = 'the model predicts the next token from the tokens before it .'

def bigram_bits(train_toks, test_toks):
    V = len(set(train_toks) | set(test_toks))
    uni, bi = {}, {}
    for a, b in zip(train_toks, train_toks[1:]):
        uni[a] = uni.get(a, 0) + 1; bi[a, b] = bi.get((a, b), 0) + 1
    return -sum(math.log2((bi.get((a, b), 0) + 1) / (uni.get(a, 0) + V)) for a, b in zip(test_toks, test_toks[1:]))

for name, cut in [('characters', list), ('words', str.split)]:
    bits = bigram_bits(cut(train), cut(test))
    n = len(cut(test)) - 1
    print(f'{name:10}  tokens={n:3}  perplexity per token={2 ** (bits / n):7.2f}   bits per character={bits / (len(test) - 1):.3f}')

## Part 2 — pass@k (section 13.2)

In [ ]:
def pass_at_k(n, c, k):
    return 1 - comb(n - c, k) / comb(n, k)

for k in (1, 5, 10):
    print(f'n=10 samples, c=3 pass: pass@{k} = {pass_at_k(10, 3, k):.3f}')

## Part 3 — How sure is a score? (section 13.6)

In [ ]:
true_acc = 0.70
for n in (50, 200, 1000):
    scores = rng.binomial(n, true_acc, 10_000) / n
    se = math.sqrt(true_acc * (1 - true_acc) / n)
    print(f'{n:5} questions: predicted ±2SE = ±{2 * se:.3f}   simulated middle 95% = [{np.quantile(scores, .025):.3f}, {np.quantile(scores, .975):.3f}]')

# Two models scoring 70% and 73% on 200 questions: is the difference real?
a, b = rng.binomial(200, 0.70, 10_000) / 200, rng.binomial(200, 0.70, 10_000) / 200
print('share of identical-model pairs differing by 3 points or more:', np.mean(np.abs(a - b) >= 0.03).round(3))

## Your turn
1. An LLM judge prefers A 65% of the time when A is shown first and 45% when it is shown second. What is the order-averaged preference?
2. How many questions would a benchmark need for ±2 standard errors to be ±1 point at 70% accuracy?

<details><summary>Answers</summary>

1. 55%. 2. $n = 4 \times 0.21 / 0.01^2 = 8{,}400$.
</details>